In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))


In [7]:
import pandas as pd


In [8]:
df = pd.read_csv("../data/processed/fraud_processed.csv")
df.head()


,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,country,hour_of_day,day_of_week,time_since_signup,txn_count_24h
0,2,2015-01-11 03:47:13,2015-02-21 10:03:37,54,FGBQNDNBETFJJ,SEO,Chrome,F,25,880217484,0,United States,10,5,3564984.0,0
1,4,2015-06-02 16:40:57,2015-09-26 21:32:16,41,MKFUIVOHLJBYN,Direct,Safari,F,38,2785906106,0,Switzerland,21,5,10039879.0,0
2,8,2015-05-28 07:53:06,2015-08-13 11:53:07,47,SCQGQALXBUQZJ,SEO,Chrome,M,25,356056736,0,United States,11,3,6667201.0,0
3,9,2015-05-16 15:58:32,2015-05-20 23:06:42,62,IEZOHXPZBIRTE,SEO,FireFox,M,21,759104706,0,Unknown,23,2,371290.0,0
4,12,2015-01-10 06:25:12,2015-03-04 20:56:37,35,MSNWCFEHKTIOY,Ads,Safari,M,19,2985180352,0,Mexico,20,2,4631485.0,0


In [9]:
DROP_COLS = ["signup_time", "purchase_time"]
df_model = df.drop(columns=DROP_COLS)


In [10]:
X = df_model.drop(columns=["class"])
y = df_model["class"]


In [11]:
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

num_features, cat_features


(['user_id',
  'purchase_value',
  'age',
  'ip_address',
  'hour_of_day',
  'day_of_week',
  'time_since_signup',
  'txn_count_24h'],
 ['device_id', 'source', 'browser', 'sex', 'country'])

In [12]:
from src.preprocessing_pipeline import PreprocessingPipeline

pipeline = PreprocessingPipeline()
transformer = pipeline.build_transformer(
    numeric_features=num_features,
    categorical_features=cat_features
)


In [13]:
from src.splitter import DataSplitter

splitter = DataSplitter()
X_train, X_test, y_train, y_test = splitter.stratified_split(X, y)


In [14]:
X_train_transformed = transformer.fit_transform(X_train)
X_test_transformed = transformer.transform(X_test)


In [15]:
from src.models import ModelFactory
from src.metrics import ModelEvaluator

lr = ModelFactory.logistic_regression()
lr.fit(X_train_transformed, y_train)

evaluator = ModelEvaluator()
lr_results = evaluator.evaluate(lr, X_test_transformed, y_test)

lr_results


{'AUC_PR': 0.6258235476016352,
 'F1': 0.6602655771195097,
 'Confusion_Matrix': array([[26944,   449],
        [ 1214,  1616]])}

In [16]:
rf = ModelFactory.random_forest(
    n_estimators=300,
    max_depth=10
)

rf.fit(X_train_transformed, y_train)
rf_results = evaluator.evaluate(rf, X_test_transformed, y_test)

rf_results


{'AUC_PR': 0.6220890789470487,
 'F1': 0.6745614035087719,
 'Confusion_Matrix': array([[27201,   192],
        [ 1292,  1538]])}

In [17]:
from src.cross_validation import CrossValidator

cv = CrossValidator(rf)
cv_results = cv.run(X_train_transformed, y_train.values)

cv_results


{'AUC_PR_mean': np.float64(0.6298575831958474),
 'AUC_PR_std': np.float64(0.01408546284383061),
 'F1_mean': np.float64(0.6836387476434207),
 'F1_std': np.float64(0.02025264809454978)}

In [18]:
comparison = pd.DataFrame([
    {"Model": "Logistic Regression", **lr_results},
    {"Model": "Random Forest", **rf_results}
])

comparison


,Model,AUC_PR,F1,Confusion_Matrix
0,Logistic Regression,0.625824,0.660266,"[[26944, 449], [1214, 1616]]"
1,Random Forest,0.622089,0.674561,"[[27201, 192], [1292, 1538]]"


## Model Selection

Logistic Regression provides a strong interpretable baseline but is limited in
capturing non-linear fraud patterns.

Random Forest achieves higher AUC-PR and F1-score, indicating superior ability
to detect fraudulent transactions under severe class imbalance.

Given the business cost of missed fraud, Random Forest is selected as the final
model despite reduced interpretability.
